In [1]:
import os
import sys
from ipywidgets import interact

root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)
 
from syn_project.utils_train import *
from syn_project.utils_notebook import *

%matplotlib widget

In [2]:
condition = "control"
data = "biased_50"
switch_epoch = 0
checkpoint_epoch = 0

n_samples = 32
show_results_fusion = True
fusion_attr_weight = 1.0
noise = 0.0

project_name = "syn"
experiment_name = get_experiment_name(condition, data, switch_epoch)


training_params = get_training_params(project_name, experiment_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

global_workspace = get_global_workspace(project_name, experiment_name, epoch=checkpoint_epoch)
data_module = get_data_module(project_name,  experiment_name)
train_samples = get_data_samples(data_module, n_samples, noise=  noise)
data_translated = get_data_translated(global_workspace, train_samples, n_samples, fusion_attr_weight, show_results_fusion)

print(data_translated["train_attr"][0:3])
print(data_translated["attr_decoded"][0:3])


/home/lucas/.cache/pypoetry/virtualenvs/alexis-n7zQ69N0-py3.11/lib/python3.11/site-packages/lightning/fabric/utilities/cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


Loading model from checkpoint: /home/lucas/gwsyn/checkpoints/syn/control_biased_50/checkpoints/last.ckpt
Loaded default weights from /home/lucas/gwsyn/checkpoints/syn/control_biased_50/checkpoints/last.ckpt
tensor([[ 1.0000,  0.0000,  0.0000,  0.1111, -0.4444, -0.4286,  0.8869, -0.4620,
          1.0000, -0.9922, -0.9922],
        [ 1.0000,  0.0000,  0.0000,  0.6667,  0.0000,  0.7143,  0.9888, -0.1493,
         -0.0039,  0.4196,  0.0980],
        [ 0.0000,  1.0000,  0.0000, -0.3333, -0.8889,  1.0000,  0.3295,  0.9441,
         -0.5451, -0.6314, -0.6392]])
tensor([[  6.3401,  -5.1332,  -4.0246,   0.1489,  -0.4390,  -0.3816,   0.9291,
          -0.4647,   0.9986,  -0.9937,  -0.9982],
        [  8.7507,  -8.8317, -10.4827,   0.6774,   0.0288,   0.7429,   0.8553,
          -0.2566,  -0.1372,   0.4277,   0.0996],
        [ -8.8612,   7.8891, -10.9918,  -0.3927,  -0.9599,   0.8853,   0.3298,
           0.8317,  -0.4880,  -0.6067,  -0.6593]], device='cuda:0')


In [11]:
train_samples[frozenset({'attr', 'v_latents'})].keys()

dict_keys(['v_latents', 'attr'])

In [32]:
unimodal_latents = global_workspace.encode_domains(train_samples)
gw_latents = global_workspace.encode(unimodal_latents)

In [48]:
gw_latent = gw_latents[frozenset({'attr', 'v_latents'})]

In [76]:
gw_latent

{'v_latents': tensor([[ 0.1820, -0.2528, -0.1174, -0.1555,  0.0681, -0.2245,  0.2249,  0.1105,
           0.0204,  0.0228, -0.0575, -0.0693],
         [ 0.0886, -0.4371, -0.0498, -0.0425,  0.0083,  0.3781, -0.2932,  0.0288,
          -0.0363, -0.2896, -0.2773,  0.0494],
         [-0.2795,  0.1438,  0.0426, -0.1606,  0.3404,  0.0793, -0.1371,  0.1674,
           0.1436,  0.0383,  0.0148, -0.0047],
         [ 0.0638,  0.0801, -0.0905,  0.2035, -0.0178, -0.1374,  0.1899, -0.1035,
          -0.0093, -0.1512,  0.5900, -0.0065],
         [ 0.0361,  0.1494, -0.5125, -0.0290, -0.0423,  0.2571, -0.1545, -0.0540,
           0.0775, -0.1541,  0.1569,  0.2754],
         [-0.4004, -0.1462,  0.0555, -0.0213, -0.2065, -0.0715,  0.0017,  0.0164,
          -0.2060, -0.4680,  0.1370,  0.0738],
         [-0.0172,  0.0156, -0.0785,  0.3137,  0.0294,  0.0729,  0.0575,  0.1026,
           0.0810, -0.2388,  0.0939, -0.0833],
         [ 0.1686, -0.0327, -0.3627, -0.0594,  0.2991,  0.2365, -0.0090, -0.0626,
  

In [106]:
a = gw_latent['v_latents'][0].detach().cpu().numpy()

In [107]:
b = {'attr': torch.Tensor([a]).to(device)}

In [108]:
c = global_workspace.decode(b, ["v_latents", "attr"])

In [109]:
c

{'attr': {'v_latents': tensor([[ 1.0773,  0.6927,  1.1383,  0.6630, -1.5869,  2.6298, -1.3090,  0.2058,
            0.0667, -0.5459,  0.7264, -0.6706]], device='cuda:0',
         grad_fn=<AddmmBackward0>),
  'attr': tensor([[ 6.1291, -4.9651, -3.9801,  0.1963, -0.4389, -0.4547,  0.8371, -0.3981,
            0.9949, -0.9958, -0.9958]], device='cuda:0', grad_fn=<CatBackward0>)}}

In [68]:
gw_latents_decoded = global_workspace.decode(gw_latent, ["v_latents", "attr"])

In [105]:
gw_latents_decoded['v_latents']['attr'][0]

tensor([ 6.1291, -4.9651, -3.9801,  0.1963, -0.4389, -0.4547,  0.8371, -0.3981,
         0.9949, -0.9958, -0.9958], device='cuda:0', grad_fn=<SelectBackward0>)

In [71]:
gw_latents_decoded["v_latents"]["v_latents"]

torch.Size([32, 12])

In [57]:
gw_latents_decoded["v_latents"]["v_latents"]
visual_module = cast(VisualLatentDomainModule, global_workspace.domain_mods["v_latents"])


In [63]:
images = visual_module.decode_images(gw_latents_decoded["v_latents"]["v_latents"]).detach().cpu()

In [64]:
images[0]

tensor([[[6.7914e-24, 2.1059e-26, 2.2135e-24,  ..., 3.1507e-32,
          1.0898e-28, 4.8817e-18],
         [3.2877e-28, 6.8785e-32, 3.3368e-30,  ..., 0.0000e+00,
          2.9345e-36, 2.6553e-23],
         [5.7619e-30, 8.9817e-35, 7.1439e-35,  ..., 0.0000e+00,
          0.0000e+00, 1.0291e-24],
         ...,
         [2.9230e-09, 2.7386e-11, 7.6751e-13,  ..., 2.8654e-29,
          9.9547e-25, 8.1884e-17],
         [6.8359e-08, 1.1705e-09, 6.3329e-11,  ..., 1.9776e-21,
          3.6204e-18, 3.4886e-12],
         [7.8488e-07, 1.0677e-07, 2.1279e-08,  ..., 9.9595e-12,
          5.8081e-10, 1.2326e-06]],

        [[1.6609e-22, 1.5059e-24, 2.5300e-22,  ..., 1.8653e-31,
          2.3434e-28, 7.9829e-18],
         [1.2484e-26, 6.6264e-30, 7.4563e-28,  ..., 1.1339e-38,
          1.2843e-35, 8.6727e-23],
         [5.6378e-28, 2.8931e-32, 3.6478e-32,  ..., 0.0000e+00,
          6.1859e-39, 2.1109e-24],
         ...,
         [2.6265e-09, 2.2202e-11, 6.0536e-13,  ..., 1.7351e-28,
          6.795

In [124]:
from typing import cast
from shimmer import GlobalWorkspace2Domains
from shimmer_ssd.modules.domains.visual import VisualLatentDomainModule


def get_decoded_image_from_gw_latent(gw_latent_array: np.ndarray, global_workspace:GlobalWorkspace2Domains, device: torch.device):
    gw_latent_tensor = {'attr': torch.Tensor([gw_latent_array]).to(device)}
    gw_latents_decoded = global_workspace.decode(gw_latent_tensor, ["v_latents", "attr"])
    
    visual_module = cast(VisualLatentDomainModule, global_workspace.domain_mods["v_latents"])
    
    images = visual_module.decode_images(gw_latents_decoded['attr']["v_latents"]).detach().cpu()

    return images[0].permute(1, 2, 0).detach().cpu().numpy()

In [113]:
get_decoded_image_from_gw_latent(a, global_workspace, device)

tensor([[[6.7913e-24, 2.1059e-26, 2.2135e-24,  ..., 3.1506e-32,
          1.0898e-28, 4.8817e-18],
         [3.2877e-28, 6.8780e-32, 3.3367e-30,  ..., 0.0000e+00,
          2.9346e-36, 2.6554e-23],
         [5.7619e-30, 8.9819e-35, 7.1437e-35,  ..., 0.0000e+00,
          0.0000e+00, 1.0291e-24],
         ...,
         [2.9230e-09, 2.7386e-11, 7.6751e-13,  ..., 2.8654e-29,
          9.9550e-25, 8.1885e-17],
         [6.8359e-08, 1.1705e-09, 6.3330e-11,  ..., 1.9776e-21,
          3.6205e-18, 3.4886e-12],
         [7.8488e-07, 1.0677e-07, 2.1279e-08,  ..., 9.9594e-12,
          5.8081e-10, 1.2326e-06]],

        [[1.6609e-22, 1.5059e-24, 2.5300e-22,  ..., 1.8654e-31,
          2.3434e-28, 7.9827e-18],
         [1.2484e-26, 6.6268e-30, 7.4561e-28,  ..., 1.1338e-38,
          1.2843e-35, 8.6728e-23],
         [5.6377e-28, 2.8929e-32, 3.6476e-32,  ..., 0.0000e+00,
          6.1861e-39, 2.1110e-24],
         ...,
         [2.6265e-09, 2.2202e-11, 6.0535e-13,  ..., 1.7351e-28,
          6.795

tensor([[[[7.4899e-25, 3.5819e-27, 1.5528e-24,  ..., 1.4940e-30,
           8.6513e-28, 9.5819e-18],
          [1.7690e-29, 3.4600e-33, 8.5310e-31,  ..., 1.6217e-37,
           5.3556e-35, 7.6971e-23],
          [2.7952e-31, 3.5965e-36, 1.8434e-35,  ..., 0.0000e+00,
           1.4405e-37, 6.9445e-24],
          ...,
          [1.7117e-08, 4.0876e-10, 2.8158e-11,  ..., 2.1327e-28,
           5.4865e-24, 1.5711e-16],
          [2.8731e-07, 9.8716e-09, 1.1641e-09,  ..., 9.3227e-21,
           1.3204e-17, 5.8644e-12],
          [2.1683e-06, 4.1045e-07, 1.1731e-07,  ..., 1.9094e-11,
           9.2662e-10, 1.2712e-06]],

         [[2.0142e-23, 3.0063e-25, 1.9895e-22,  ..., 7.6480e-30,
           1.8224e-27, 1.6627e-17],
          [6.7728e-28, 3.5863e-31, 2.1217e-28,  ..., 1.1681e-36,
           2.1808e-34, 2.6874e-22],
          [2.9792e-29, 1.3403e-33, 1.0994e-32,  ..., 0.0000e+00,
           3.7065e-37, 1.5077e-23],
          ...,
          [1.3854e-08, 2.7307e-10, 1.9490e-11,  ..., 1.2187

In [117]:
a

array([ 0.18197833, -0.25282288, -0.11736982, -0.15546931,  0.06807819,
       -0.22454223,  0.22493014,  0.11052177,  0.02038486,  0.02281225,
       -0.05748973, -0.06934198], dtype=float32)

In [126]:
# On définit les réglages des 12 curseurs (-1 à 1, pas de 0.1)
sliders = {f'v{i+1}': (-1.0, 1.0, 0.1) for i in range(12)}

@interact(
    **sliders
)
def play_with_gw(**values):
    latent_vector = np.array([values[k] for k in sorted(values.keys())], dtype=np.float32)
    img = get_decoded_image_from_gw_latent(latent_vector, global_workspace, device)
    plt.imshow(img)
    plt.axis('off') # Souvent plus joli pour des images générées
    plt.show()


interactive(children=(FloatSlider(value=0.0, description='v1', max=1.0, min=-1.0), FloatSlider(value=0.0, desc…